# Análise Exploratória dos Dados — Booklog AI

## Integrantes
- Gabriel Nottoli Buck — RA 10425384 — e-mail não fornecido
- Julia Andrade — RA 10427828 — e-mail não fornecido
- Joao Vitor Rocha Miranda — RA 10427273 — e-mail não fornecido

## Descrição
Notebook da N1 destinado à análise exploratória e preparação do dataset original do projeto de recomendação personalizada de livros.

## Histórico de alterações

| Data | Autor | Alteração |
|---|---|---|
| 15/09/2026 | Codex, a pedido do grupo | Estrutura inicial, validação e preparação |

**Situação: dataset real ainda não disponibilizado. Nenhum resultado empírico foi produzido.** As saídas só serão geradas com entradas reais válidas. Os testes artificiais não são executados como parte da EDA.

## 1. Imports e ambiente
Execute a partir da raiz do projeto ou de `notebooks/`. Reinicie o kernel e execute todas as células ao trocar as entradas. Saídas antigas em disco não são apagadas automaticamente.

In [ ]:
from pathlib import Path
import sys
import json
import hashlib
import platform
from datetime import datetime, timezone
from importlib.metadata import version
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'src/data.py').is_file() and (p / 'data/README.md').is_file()), None)
if ROOT is None:
    raise RuntimeError('Abra o notebook dentro do repositório Booklog AI.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.data import load_csv, audit, prepare, summarize
VERSIONS = {name: version(name) for name in ['pandas', 'numpy', 'matplotlib']}
print('Python:', platform.python_version(), '| Bibliotecas:', VERSIONS)

## 2. Carregamento de dados
Sem as duas entradas, as etapas seguintes são ignoradas e nada é exportado. Um arquivo presente mas inválido será rejeitado. Não há geração automática de dataset.

In [ ]:
paths = {name: ROOT / 'data/raw' / f'{name}.csv' for name in ['ratings', 'books']}
AVAILABLE = all(path.is_file() for path in paths.values())
VALID = False
ratings_raw = books_raw = ratings = books = None
if AVAILABLE:
    ratings_raw = load_csv(paths['ratings'], 'ratings')
    books_raw = load_csv(paths['books'], 'books')
    print('Entradas carregadas; conteúdo ainda precisa ser validado.')
else:
    print('PENDENTE: disponibilizar os dois CSVs reais revisados em data/raw.')

## 3. Validação estrutural
A carga exige as colunas exatas e preserva identificadores como texto. A validação semântica ocorre após mostrar ausências e duplicidades da entrada normalizada.

In [ ]:
if AVAILABLE:
    for name, frame in [('ratings', ratings_raw), ('books', books_raw)]:
        print(name, '| linhas:', len(frame), '| colunas:', list(frame.columns))

## 4. Valores nulos
Ausências são calculadas depois de retirar espaços externos e normalizar campos vazios. Não mostrar linhas individuais nesta auditoria.

In [ ]:
if AVAILABLE:
    for name, frame in [('ratings', ratings_raw), ('books', books_raw)]:
        print(name)
        display(audit(frame))

## 5. Duplicidades e integridade
Cópias idênticas podem ser removidas; conflitos são bloqueados. Não excluir usuários pouco ativos nem notas extremas válidas automaticamente.

In [ ]:
if AVAILABLE:
    print('Cópias exatas de avaliações:', ratings_raw.duplicated().sum())
    print('Cópias exatas de livros:', books_raw.duplicated().sum())
    print('Repetições de par usuário–livro:', ratings_raw.duplicated(['user_id', 'book_id']).sum())
    ratings, books, removed = prepare(ratings_raw, books_raw)
    VALID = True
    print('Validação concluída. Cópias idênticas removidas:', removed)

## 6. Estatísticas do recorte validado
As seções seguintes usam dados sem cópias exatas. Usuários são apenas os observados nas avaliações; não representam o total de contas do Booklog.

In [ ]:
if VALID:
    summary = summarize(ratings, books)
    display(summary.to_frame())
    display(ratings.rating.describe().to_frame('nota'))

## 7. Distribuição de notas
A escala aceita frações de 1 a 5. A distribuição mostra valores efetivamente observados.

In [ ]:
if VALID:
    figures = {}
    rating_counts = ratings.rating.value_counts().sort_index().rename('avaliacoes')
    fig, ax = plt.subplots(figsize=(8, 4))
    rating_counts.plot.bar(ax=ax, color='#405b78')
    ax.set(xlabel='Nota', ylabel='Avaliações', title='Distribuição das notas observadas')
    fig.tight_layout(); figures['distribuicao_notas'] = fig
    plt.show()

## 8. Atividade por usuário
A tabela fica local até revisão. O gráfico agrega quantos leitores possuem cada quantidade de avaliações, sem exibir identificadores.

In [ ]:
if VALID:
    by_user = ratings.groupby('user_id').size().rename('avaliacoes')
    display(by_user.describe().to_frame())
    fig, ax = plt.subplots(figsize=(8, 4))
    by_user.value_counts().sort_index().plot.bar(ax=ax, color='#405b78')
    ax.set(xlabel='Avaliações por usuário', ylabel='Usuários', title='Atividade dos leitores')
    fig.tight_layout(); figures['atividade_usuarios'] = fig
    plt.show()

## 9. Atividade por livro, gêneros e concentração
Livros do catálogo sem avaliação entram com contagem zero de interações, nunca como nota zero. A curva acumulada mostra concentração. Gêneros contam livros do catálogo, sem ponderação por avaliações; obras multigênero podem aparecer em mais de uma categoria. Preservar a taxonomia da fonte e discutir variantes de nomes.

In [ ]:
if VALID:
    by_book = ratings.groupby('book_id').size().reindex(books.book_id, fill_value=0).rename('avaliacoes')
    display(by_book.describe().to_frame())
    ranked = by_book.sort_values(ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(1, len(ranked) + 1), ranked.cumsum() / ranked.sum())
    ax.set(xlabel='Livros, do mais ao menos avaliado', ylabel='Fração acumulada das avaliações',
           title='Concentração de avaliações no catálogo', ylim=(0, 1.05))
    fig.tight_layout(); figures['concentracao_livros'] = fig
    plt.show()
    tokens = books[['book_id', 'genres']].dropna().copy()
    tokens['genres'] = tokens.genres.str.split('|', regex=False)
    tokens = tokens.explode('genres')
    tokens['genres'] = tokens.genres.str.strip()
    genre_counts = tokens.drop_duplicates(['book_id', 'genres']).groupby('genres').size().sort_values(ascending=False).rename('livros')
    print('Livros sem gênero:', books.genres.isna().sum())
    display(genre_counts.to_frame())

## 10. Cobertura e esparsidade
Cobertura = livros avaliados / catálogo. Esparsidade = 1 − pares únicos / (usuários observados × livros). Reportar tanto o catálogo inteiro quanto apenas livros avaliados. Não materializar matriz densa. Valores ausentes da matriz significam desconhecimento.

In [ ]:
if VALID:
    display(summary.loc[['cobertura_catalogo', 'esparsidade_catalogo',
                         'esparsidade_livros_avaliados']].to_frame())

## 11. Preparação e auditoria
A preparação já foi aplicada na seção 5 para a EDA válida: normalização de espaços/vazios, remoção de cópias exatas, validação, nota numérica e ordenação determinística. Ausências opcionais são preservadas. Não há imputação, engenharia de atributos aprendida nem divisão de treino/teste; estas dependem do protocolo N2.

In [ ]:
if VALID:
    preparation = pd.DataFrame({
        'entrada': [len(ratings_raw), len(books_raw)],
        'copias_removidas': [removed['ratings'], removed['books']],
        'saida': [len(ratings), len(books)],
    }, index=['ratings', 'books'])
    display(preparation)

## 12. Exportação local
Exportar somente dados válidos. CSVs de entrada permanecem intactos. Dados derivados, gráficos e tabelas ficam ignorados pelo Git até revisão. O manifesto identifica a execução, mas não comprova consentimento/origem. Antes de publicar, conferir também riscos de reidentificação nas saídas.

In [ ]:
if VALID:
    processed = ROOT / 'data/processed'
    tables = ROOT / 'results/tables'
    figures_dir = ROOT / 'results/figures'
    for folder in [processed, tables, figures_dir]:
        folder.mkdir(parents=True, exist_ok=True)
    ratings.to_csv(processed / 'ratings.csv', index=False)
    books.to_csv(processed / 'books.csv', index=False)
    outputs = {'resumo': summary, 'notas': rating_counts, 'por_usuario': by_user,
               'por_livro': by_book, 'generos': genre_counts, 'preparacao': preparation,
               'ausencias_ratings': audit(ratings_raw), 'ausencias_books': audit(books_raw)}
    for name, frame in outputs.items():
        frame.to_csv(tables / f'{name}.csv')
    for name, fig in figures.items():
        fig.savefig(figures_dir / f'{name}.png', dpi=160, bbox_inches='tight')
        plt.close(fig)
    def sha256(path):
        return hashlib.sha256(path.read_bytes()).hexdigest()
    manifest = {
        'executado_em_utc': datetime.now(timezone.utc).isoformat(),
        'python': platform.python_version(), 'bibliotecas': VERSIONS,
        'entrada_sha256': {name: sha256(path) for name, path in paths.items()},
        'saida_sha256': {name: sha256(processed / f'{name}.csv') for name in paths},
        'codigo_sha256': sha256(ROOT / 'src/data.py'),
        'copias_removidas': removed,
        'contrato': 'data/README.md, versão 1; escala 1 a 5',
        'origem': 'Consultar documentação real da coleta; não inferida pelo código',
    }
    (processed / 'manifest.json').write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding='utf-8')
    print('Exportação local concluída. Revisar antes de publicar.')

## 13. Conclusões da EDA
**Sem dados, não há conclusões empíricas.** Após execução real, o grupo deverá interpretar: tamanho e recrutamento da amostra; atividade/sobreposição; notas; ausências e perdas; cobertura de metadados; concentração; esparsidade; adequação do experimento e limitações de generalização. Não inferir automaticamente que o modelo é viável ou que superará o baseline.

Substituir este roteiro por conclusões sustentadas pelas saídas da coleta quando disponíveis. Não inserir números hipotéticos. A metodologia futura está em `docs/metodologia.md`.

In [ ]:
if not VALID:
    print('EDA REAL PENDENTE — nenhum resultado ou arquivo derivado foi gerado nesta execução.')
else:
    print('EDA calculada. Interpretação acadêmica pelo grupo ainda necessária.')